# Predict News Categories 
Use the fine-tuned BERT model to classify any news headline.

**Options**
- Single headline  
- Batch of headlines  
- Visualise confidence scores  


## 1 · Setup

In [3]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import BertTokenizerFast, BertForSequenceClassification

MODEL_DIR  = "models/bert_ag_news"
MAX_LENGTH = 128
LABEL_NAMES = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
EMOJI  = {"World": "🌍", "Sports": "⚽", "Business": "💼", "Sci/Tech": "🔬"}
COLORS = {"World": "#3B82F6", "Sports": "#10B981",
          "Business": "#F59E0B", "Sci/Tech": "#8B5CF6"}

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizerFast.from_pretrained(MODEL_DIR)
model     = BertForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval().to(device)
print(f"Model ready on {device} ✅")


OSError: Error no file named model.safetensors, or pytorch_model.bin, found in directory models/bert_ag_news.

## 2 · Prediction Function

In [ ]:
def predict(headlines):
    """
    Classify one or more headlines.
    Returns list of dicts: {label, confidence, all_probs}
    """
    if isinstance(headlines, str):
        headlines = [headlines]

    inputs = tokenizer(
        headlines, return_tensors="pt",
        truncation=True, max_length=MAX_LENGTH, padding=True,
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    probs_all = torch.softmax(logits, dim=-1).cpu().numpy()
    results = []
    for probs in probs_all:
        idx = int(np.argmax(probs))
        results.append({
            "label":      LABEL_NAMES[idx],
            "confidence": float(probs[idx]),
            "all_probs":  {LABEL_NAMES[i]: float(probs[i]) for i in range(4)},
        })
    return results if len(results) > 1 else results[0]


## 3 · Single Headline

In [ ]:
headline = "NASA's James Webb Telescope captures the earliest galaxies ever observed"

result = predict(headline)
emoji  = EMOJI[result['label']]
print(f"Headline  : {headline}")
print(f"Category  : {emoji} {result['label']}")
print(f"Confidence: {result['confidence']*100:.1f}%")
print("\nAll scores:")
for label, prob in sorted(result["all_probs"].items(), key=lambda x: -x[1]):
    bar = "█" * int(prob * 30) + "░" * (30 - int(prob * 30))
    print(f"  {label:<12} {bar}  {prob*100:5.1f}%")


## 4 · Visualise Confidence Scores

In [ ]:
def plot_confidence(headline, result):
    labels  = list(result["all_probs"].keys())
    values  = list(result["all_probs"].values())
    colors  = [COLORS[l] for l in labels]
    pred    = result["label"]

    fig, ax = plt.subplots(figsize=(8, 3.5))
    bars = ax.barh(labels, values, color=colors, height=0.5, edgecolor="white")
    ax.set_xlim(0, 1.1)
    ax.set_xlabel("Probability", fontsize=11)
    ax.set_title(f"'{headline[:70]}…'", fontsize=10, style="italic")
    ax.axvline(0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)

    for bar, v, lbl in zip(bars, values, labels):
        weight = "bold" if lbl == pred else "normal"
        ax.text(v + 0.01, bar.get_y() + bar.get_height()/2,
                f"{v*100:.1f}%", va="center", fontsize=11, fontweight=weight)

    ax.text(0.99, -0.18, f"Predicted: {EMOJI[pred]} {pred}",
            transform=ax.transAxes, ha="right", fontsize=12, fontweight="bold",
            color=COLORS[pred])
    plt.tight_layout()
    plt.show()

plot_confidence(headline, result)


## 5 · Batch Prediction

In [ ]:
headlines = [
    "WHO warns of new respiratory disease outbreak in Southeast Asia",
    "Manchester City defeats Real Madrid 3-1 in Champions League final",
    "Federal Reserve raises interest rates by 25 basis points",
    "NASA's Perseverance rover discovers ancient river delta on Mars",
    "Apple reports record quarterly revenue of $120 billion",
    "UN Security Council holds emergency session on border conflict",
    "Lionel Messi retires from international football at 38",
    "OpenAI releases GPT-5 with real-time video understanding",
]

results = predict(headlines)

import pandas as pd
rows = []
for h, r in zip(headlines, results):
    rows.append({
        "Headline"  : h[:75] + ("…" if len(h)>75 else ""),
        "Category"  : f"{EMOJI[r['label']]} {r['label']}",
        "Confidence": f"{r['confidence']*100:.1f}%",
    })
df = pd.DataFrame(rows)
display(df.style.hide(axis="index"))


## 6 · Multi-headline Confidence Heatmap

In [ ]:
import seaborn as sns

short_labels = [h[:45]+"…" if len(h)>45 else h for h in headlines]
matrix = np.array([[r["all_probs"][LABEL_NAMES[i]] for i in range(4)]
                   for r in results])

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(matrix, annot=True, fmt=".2f", cmap="YlOrRd",
            xticklabels=list(LABEL_NAMES.values()),
            yticklabels=short_labels,
            linewidths=0.4, ax=ax,
            vmin=0, vmax=1)
ax.set_title("Prediction Confidence Heatmap", fontsize=14, fontweight="bold")
ax.set_xlabel("Category"); ax.set_ylabel("")
plt.tight_layout()
plt.savefig("outputs/confidence_heatmap.png", dpi=130)
plt.show()


## 7 · Try Your Own Headline ✏️

In [2]:
# ── Change this headline and re-run the cell ──────────────────
my_headline = "Type your news headline here"

r = predict(my_headline)
print(f"Category   : {EMOJI[r['label']]} {r['label']}")
print(f"Confidence : {r['confidence']*100:.1f}%")
plot_confidence(my_headline, r)


NameError: name 'predict' is not defined